In [43]:
import numpy as np
import pandas as pd
import psycopg

In [44]:
conn = psycopg.connect("dbname=dailyedge_development")

In [45]:
predictions = pd.read_csv("../data/nq_first_to_100_predictions_full.csv")

print("Shape:", predictions.shape)
print("Columns:", predictions.columns.tolist())
display(predictions.head())

Shape: (508, 9)
Columns: ['Date', 'Day', 'Bias', 'Confidence', 'Auction Direction', 'Context', 'Result', 'Correct', 'Notes']


,Date,Day,Bias,Confidence,Auction Direction,Context,Result,Correct,Notes
0,2024-09-02,Monday,Long,61%,Moderate Up,Trend Continuation,Invalid,NaN,Positive ORG followed an overnight sell-side s...
1,2024-09-03,Tuesday,Short,63%,Strong Down,Trend Continuation,Short,True,Large negative ORG accompanied a persistent de...
2,2024-09-04,Wednesday,Short,66%,Strong Down,Trend Continuation,Long,False,Large negative ORG followed an exceptionally w...
3,2024-09-05,Thursday,Short,59%,Moderate Down,Balance,Long,False,Negative ORG developed within a choppy overnig...
4,2024-09-06,Friday,Long,62%,Moderate Up,Exhaustion,Short,False,Nearly flat ORG followed a broad overnight ran...


In [46]:
predictions["Date"] = pd.to_datetime(
    predictions["Date"],
    format="%Y-%m-%d",
    errors="raise"
)

print("Earliest date:", predictions["Date"].min().date())
print("Latest date:", predictions["Date"].max().date())
print("Unique dates:", predictions["Date"].nunique())
print("Duplicate dates:", predictions["Date"].duplicated().sum())
print("\nCorrect value counts:")
print(predictions["Correct"].value_counts(dropna=False))

Earliest date: 2024-09-02
Latest date: 2026-08-21
Unique dates: 507
Duplicate dates: 0

Correct value counts:
Correct
True     259
False    223
NaN       26
Name: count, dtype: int64


In [47]:
study_start = pd.Timestamp("2024-09-02")
study_end = pd.Timestamp("2026-08-21")

study_predictions = (
    predictions.loc[
        predictions["Date"].between(study_start, study_end)
    ]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Study rows:", len(study_predictions))
print("Unique study dates:", study_predictions["Date"].nunique())
print("Earliest study date:", study_predictions["Date"].min().date())
print("Latest study date:", study_predictions["Date"].max().date())
print("\nCorrect value counts:")
print(study_predictions["Correct"].value_counts(dropna=False))

Study rows: 507
Unique study dates: 507
Earliest study date: 2024-09-02
Latest study date: 2026-08-21

Correct value counts:
Correct
True     259
False    223
NaN       25
Name: count, dtype: int64


In [48]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT timestamp, open, high, low, close
        FROM CANDLES
        WHERE timestamp::date BETWEEN %s AND %s
          AND timestamp::time BETWEEN TIME '08:30:00' AND TIME '15:15:00'
        ORDER BY timestamp
        """,
        (study_start.date(), study_end.date())
    )

    candle_rows = cur.fetchall()
    candle_columns = [column.name for column in cur.description]

candles = pd.DataFrame(candle_rows, columns=candle_columns)
candles["timestamp"] = pd.to_datetime(candles["timestamp"])
candles["Date"] = candles["timestamp"].dt.normalize()

print("Candle rows:", len(candles))
print("Candle dates:", candles["Date"].nunique())
print("Earliest candle:", candles["timestamp"].min())
print("Latest candle:", candles["timestamp"].max())
print("\nCandles per session:")
print(candles.groupby("Date").size().describe())

Candle rows: 202598
Candle dates: 508
Earliest candle: 2024-09-02 08:30:00
Latest candle: 2026-08-21 15:15:00

Candles per session:
count    508.000000
mean     398.814961
std       36.498753
min      210.000000
25%      406.000000
50%      406.000000
75%      406.000000
max      406.000000
dtype: float64


In [49]:
prediction_dates = set(study_predictions["Date"])
candle_dates = set(candles["Date"])

matched_dates = prediction_dates & candle_dates
prediction_only_dates = prediction_dates - candle_dates
candle_only_dates = candle_dates - prediction_dates

matched_predictions = (
    study_predictions.loc[study_predictions["Date"].isin(matched_dates)]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Prediction dates:", len(prediction_dates))
print("Candle dates:", len(candle_dates))
print("Matched dates:", len(matched_dates))
print(
    "Prediction-only:",
    [date.strftime("%Y-%m-%d") for date in sorted(prediction_only_dates)]
)
print(
    "Candle-only:",
    [date.strftime("%Y-%m-%d") for date in sorted(candle_only_dates)]
)
print("\nMatched Correct value counts:")
print(matched_predictions["Correct"].value_counts(dropna=False))

Prediction dates: 507
Candle dates: 508
Matched dates: 505
Prediction-only: ['2025-01-09', '2026-04-03']
Candle-only: ['2025-01-20', '2025-03-04', '2026-04-24']

Matched Correct value counts:
Correct
True     259
False    223
NaN       23
Name: count, dtype: int64


In [50]:
correct_predictions = (
    matched_predictions.loc[matched_predictions["Correct"].eq(True)]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

correct_predictions["Calendar Day"] = correct_predictions["Date"].dt.day_name()

print("Correctly predicted matched sessions:", len(correct_predictions))

print("\nBias counts:")
print(correct_predictions["Bias"].value_counts(dropna=False))

print("\nWeekday counts:")
print(
    correct_predictions["Day"]
    .value_counts()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"])
)

day_mismatches = correct_predictions.loc[
    correct_predictions["Day"] != correct_predictions["Calendar Day"],
    ["Date", "Day", "Calendar Day"]
]

print("\nRecorded-day/calendar-day mismatches:", len(day_mismatches))
display(day_mismatches)

Correctly predicted matched sessions: 259

Bias counts:
Bias
Long     130
Short    129
Name: count, dtype: int64

Weekday counts:
Day
Monday       50
Tuesday      55
Wednesday    59
Thursday     40
Friday       55
Name: count, dtype: int64

Recorded-day/calendar-day mismatches: 0


,Date,Day,Calendar Day


In [51]:
def reverse_bias(bias):
    return {"Long": "Short", "Short": "Long"}.get(bias, bias)

reversed_predictions = matched_predictions.copy()

is_thursday = reversed_predictions["Day"].eq("Thursday")

reversed_predictions["Bias"] = np.where(
    is_thursday,
    reversed_predictions["Bias"].map(reverse_bias),
    reversed_predictions["Bias"],
)

reversed_predictions["Correct"] = np.where(
    is_thursday & reversed_predictions["Correct"].notna(),
    ~reversed_predictions["Correct"].astype(bool),
    reversed_predictions["Correct"],
)

reversed_correct_predictions = (
    reversed_predictions.loc[reversed_predictions["Correct"].eq(True)]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Reversed-correct matched sessions:", len(reversed_correct_predictions))

Reversed-correct matched sessions: 275


In [52]:
stop_target_pairs = [
    (15, 25),
    (25, 25),
    (25, 50),
    (35, 35),
    (35, 70),
    (50, 50),
    (50, 70),
    (50, 80),
    (50, 100),
    (50, 150),
    (75, 100),
    (75, 150),
    (100, 150),
]

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
]

print("Stop/target pairs:", stop_target_pairs)
print("Weekday order:", weekday_order)

Stop/target pairs: [(15, 25), (25, 25), (25, 50), (35, 35), (35, 70), (50, 50), (50, 70), (50, 80), (50, 100), (50, 150), (75, 100), (75, 150), (100, 150)]
Weekday order: ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']


In [53]:
def evaluate_predicted_continuous_trail(
    session_candles,
    bias,
    stop_distance,
    target_distance,
):
    session = (
        session_candles[
            ["timestamp", "open", "high", "low", "close"]
        ]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    if session.empty:
        raise ValueError("Session contains no candles.")

    if bias not in {"Long", "Short"}:
        raise ValueError(f"Unexpected Bias value: {bias!r}")

    if session[["open", "high", "low", "close"]].isna().any().any():
        raise ValueError("Session contains missing OHLC values.")

    opening_price = float(session.loc[0, "open"])

    if bias == "Long":
        target = opening_price + target_distance
        highest_price = opening_price
        trailing_stop = opening_price - stop_distance

        for row in session.itertuples(index=False):
            open_price = float(row.open)
            high = float(row.high)
            low = float(row.low)

            old_stop = trailing_stop

            if open_price <= old_stop:
                return "Failure"

            new_highest = max(highest_price, high)
            new_stop = new_highest - stop_distance

            old_stop_hit = low <= old_stop
            new_stop_hit = low <= new_stop
            target_hit = high >= target

            if old_stop_hit:
                adverse_first = "Failure"
            elif target_hit:
                adverse_first = "Success"
            else:
                adverse_first = "Continue"

            if target_hit:
                favorable_first = "Success"
            elif new_stop_hit:
                favorable_first = "Failure"
            else:
                favorable_first = "Continue"

            if adverse_first == favorable_first:
                if adverse_first in {"Success", "Failure"}:
                    return adverse_first

            if {adverse_first, favorable_first} == {
                "Success",
                "Failure",
            }:
                return "Unknown/Ambiguous"

            highest_price = new_highest
            trailing_stop = new_stop

    else:
        target = opening_price - target_distance
        lowest_price = opening_price
        trailing_stop = opening_price + stop_distance

        for row in session.itertuples(index=False):
            open_price = float(row.open)
            high = float(row.high)
            low = float(row.low)

            old_stop = trailing_stop

            if open_price >= old_stop:
                return "Failure"

            new_lowest = min(lowest_price, low)
            new_stop = new_lowest + stop_distance

            old_stop_hit = high >= old_stop
            new_stop_hit = high >= new_stop
            target_hit = low <= target

            if old_stop_hit:
                adverse_first = "Failure"
            elif target_hit:
                adverse_first = "Success"
            else:
                adverse_first = "Continue"

            if target_hit:
                favorable_first = "Success"
            elif new_stop_hit:
                favorable_first = "Failure"
            else:
                favorable_first = "Continue"

            if adverse_first == favorable_first:
                if adverse_first in {"Success", "Failure"}:
                    return adverse_first

            if {adverse_first, favorable_first} == {
                "Success",
                "Failure",
            }:
                return "Unknown/Ambiguous"

            lowest_price = new_lowest
            trailing_stop = new_stop

    return "Neither"


print("Bias-driven continuous trailing evaluator defined.")

Bias-driven continuous trailing evaluator defined.


In [54]:
continuous_test_cases = {
    "Success": pd.DataFrame(
        {
            "timestamp": pd.to_datetime(
                ["2026-01-01 08:30", "2026-01-01 08:31"]
            ),
            "open": [100.0, 110.0],
            "high": [110.0, 126.0],
            "low": [100.0, 112.0],
            "close": [110.0, 125.0],
        }
    ),
    "Failure": pd.DataFrame(
        {
            "timestamp": pd.to_datetime(["2026-01-01 08:30"]),
            "open": [100.0],
            "high": [105.0],
            "low": [84.0],
            "close": [90.0],
        }
    ),
    "Unknown/Ambiguous": pd.DataFrame(
        {
            "timestamp": pd.to_datetime(
                ["2026-01-01 08:30", "2026-01-01 08:31"]
            ),
            "open": [100.0, 110.0],
            "high": [110.0, 126.0],
            "low": [100.0, 94.0],
            "close": [110.0, 105.0],
        }
    ),
}

continuous_test_results = {
    name: evaluate_predicted_continuous_trail(
        session_candles=session,
        bias="Long",
        stop_distance=15,
        target_distance=25,
    )
    for name, session in continuous_test_cases.items()
}

assert continuous_test_results == {
    "Success": "Success",
    "Failure": "Failure",
    "Unknown/Ambiguous": "Unknown/Ambiguous",
}

display(
    pd.Series(
        continuous_test_results,
        name="Returned outcome",
    ).to_frame()
)

,Returned outcome
Success,Success
Failure,Failure
Unknown/Ambiguous,Unknown/Ambiguous


In [55]:
candle_sessions = {
    date: session.copy()
    for date, session in candles.groupby("Date", sort=True)
}

continuous_records = []

for prediction in correct_predictions.itertuples(index=False):
    session = candle_sessions[prediction.Date]

    for stop_distance, target_distance in stop_target_pairs:
        outcome = evaluate_predicted_continuous_trail(
            session_candles=session,
            bias=prediction.Bias,
            stop_distance=stop_distance,
            target_distance=target_distance,
        )

        continuous_records.append(
            {
                "Date": prediction.Date,
                "Day": prediction.Day,
                "Bias": prediction.Bias,
                "Stop": stop_distance,
                "Target": target_distance,
                "Pair": f"{stop_distance}/{target_distance}",
                "Outcome": outcome,
            }
        )

continuous_results = pd.DataFrame(continuous_records)

pair_order = [
    f"{stop_distance}/{target_distance}"
    for stop_distance, target_distance in stop_target_pairs
]

continuous_outcome_counts = (
    pd.crosstab(
        continuous_results["Pair"],
        continuous_results["Outcome"],
    )
    .reindex(index=pair_order, fill_value=0)
    .reindex(
        columns=[
            "Success",
            "Failure",
            "Unknown/Ambiguous",
            "Neither",
        ],
        fill_value=0,
    )
)

continuous_outcome_counts["Total"] = (
    continuous_outcome_counts.sum(axis=1)
)

print("Result rows:", len(continuous_results))
print("Expected rows:", len(correct_predictions) * len(stop_target_pairs))
print("Unique dates:", continuous_results["Date"].nunique())

display(continuous_outcome_counts)


Result rows: 3367
Expected rows: 3367
Unique dates: 259


Outcome,Success,Failure,Unknown/Ambiguous,Neither,Total
Pair,,,,,
15/25,118,105,36,0,259
25/25,160,79,20,0,259
25/50,105,146,8,0,259
35/35,168,87,4,0,259
35/70,97,159,3,0,259
50/50,178,80,1,0,259
50/70,148,110,1,0,259
50/80,120,135,4,0,259
50/100,82,174,3,0,259


In [56]:
continuous_resolved = continuous_results.loc[
    continuous_results["Outcome"].isin(["Success", "Failure"])
].copy()

continuous_resolved["Success"] = (
    continuous_resolved["Outcome"].eq("Success")
)

continuous_weekday_summary = (
    continuous_resolved
    .groupby(["Pair", "Day"], as_index=False)
    .agg(
        Successes=("Success", "sum"),
        Resolved=("Success", "size"),
    )
)

continuous_weekday_summary["Success Rate"] = (
    continuous_weekday_summary["Successes"]
    / continuous_weekday_summary["Resolved"]
    * 100
)

continuous_rate_table = (
    continuous_weekday_summary
    .pivot(
        index="Pair",
        columns="Day",
        values="Success Rate",
    )
    .reindex(index=pair_order, columns=weekday_order)
)

continuous_sample_table = (
    continuous_weekday_summary
    .pivot(
        index="Pair",
        columns="Day",
        values="Resolved",
    )
    .reindex(index=pair_order, columns=weekday_order)
    .astype("Int64")
)

continuous_rate_display = continuous_rate_table.map(
    lambda value: f"{value:.2f}%" if pd.notna(value) else "—"
)

continuous_rate_display.index.name = "Pair"
continuous_sample_table.index.name = "Pair"

print("CONTINUOUS TRAILING — SUCCESS RATE")
display(continuous_rate_display)

print("CONTINUOUS TRAILING — RESOLVED SAMPLE SIZE")
display(continuous_sample_table)

CONTINUOUS TRAILING — SUCCESS RATE


Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,46.34%,51.06%,60.00%,57.14%,48.89%
25/25,63.83%,66.00%,71.93%,58.33%,71.43%
25/50,33.33%,40.38%,50.88%,35.90%,45.45%
35/35,70.00%,65.45%,70.18%,60.53%,61.82%
35/70,44.90%,25.93%,48.28%,30.00%,38.18%
50/50,76.00%,69.09%,71.19%,57.50%,68.52%
50/70,66.00%,52.73%,62.71%,45.00%,57.41%
50/80,60.00%,47.17%,45.61%,37.50%,43.64%
50/100,40.82%,27.78%,29.31%,22.50%,38.18%


CONTINUOUS TRAILING — RESOLVED SAMPLE SIZE


Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,41,47,55,35,45
25/25,47,50,57,36,49
25/50,48,52,57,39,55
35/35,50,55,57,38,55
35/70,49,54,58,40,55
50/50,50,55,59,40,54
50/70,50,55,59,40,54
50/80,50,53,57,40,55
50/100,49,54,58,40,55


In [57]:
continuous_records_reversed = []

for prediction in reversed_correct_predictions.itertuples(index=False):
    session = candle_sessions[prediction.Date]

    for stop_distance, target_distance in stop_target_pairs:
        outcome = evaluate_predicted_continuous_trail(
            session_candles=session,
            bias=prediction.Bias,
            stop_distance=stop_distance,
            target_distance=target_distance,
        )

        continuous_records_reversed.append(
            {
                "Date": prediction.Date,
                "Day": prediction.Day,
                "Bias": prediction.Bias,
                "Stop": stop_distance,
                "Target": target_distance,
                "Pair": f"{stop_distance}/{target_distance}",
                "Outcome": outcome,
            }
        )

continuous_results_reversed = pd.DataFrame(continuous_records_reversed)

In [58]:
total_resolved_predictions = (
    matched_predictions["Correct"].notna().sum()
)

continuous_total_success = (
    continuous_results
    .loc[continuous_results["Outcome"].eq("Success")]
    .groupby("Pair")
    .size()
    .reindex(pair_order, fill_value=0)
    .rename("Correct and Clean")
    .to_frame()
)

continuous_total_success["Total Resolved Predictions"] = (
    total_resolved_predictions
)

continuous_total_success["Total Success Rate"] = (
    continuous_total_success["Correct and Clean"]
    / continuous_total_success["Total Resolved Predictions"]
    * 100
)

continuous_total_success["Total Success Rate"] = (
    continuous_total_success["Total Success Rate"]
    .map(lambda value: f"{value:.2f}%")
)

continuous_total_success.index.name = "Pair"

display(continuous_total_success)

,Correct and Clean,Total Resolved Predictions,Total Success Rate
Pair,,,
15/25,118,482,24.48%
25/25,160,482,33.20%
25/50,105,482,21.78%
35/35,168,482,34.85%
35/70,97,482,20.12%
50/50,178,482,36.93%
50/70,148,482,30.71%
50/80,120,482,24.90%
50/100,82,482,17.01%


Thursdays Reversed

In [59]:
total_resolved_predictions = (
    matched_predictions["Correct"].notna().sum()
)

continuous_total_success_reversed = (
    continuous_results_reversed
    .loc[continuous_results_reversed["Outcome"].eq("Success")]
    .groupby("Pair")
    .size()
    .reindex(pair_order, fill_value=0)
    .rename("Correct and Clean")
    .to_frame()
)

continuous_total_success_reversed["Total Resolved Predictions"] = (
    total_resolved_predictions
)

continuous_total_success_reversed["Total Success Rate"] = (
    continuous_total_success_reversed["Correct and Clean"]
    / continuous_total_success_reversed["Total Resolved Predictions"]
    * 100
)

continuous_total_success_reversed["Total Success Rate"] = (
    continuous_total_success_reversed["Total Success Rate"]
    .map(lambda value: f"{value:.2f}%")
)

continuous_total_success_reversed.index.name = "Pair"

display(continuous_total_success_reversed)

,Correct and Clean,Total Resolved Predictions,Total Success Rate
Pair,,,
15/25,118,482,24.48%
25/25,172,482,35.68%
25/50,111,482,23.03%
35/35,180,482,37.34%
35/70,103,482,21.37%
50/50,193,482,40.04%
50/70,160,482,33.20%
50/80,130,482,26.97%
50/100,93,482,19.29%
